In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')
%cd "/content/gdrive/MyDrive/BDS Project - Irvin, Chris, Jerry"
%ls

Mounted at /content/gdrive/
/content/gdrive/.shortcut-targets-by-id/1FWooCKeslO8xilmPVIjcGiQScaBcbj9z/BDS Project - Irvin, Chris, Jerry
 10k_filings/
 10k_filings_pdf/
 10k_filings_txt/
 10k_filings_url.txt
 8k_filings/
 8k_filings_pdf/
 8k_filings_url.txt
 alternate_chatbot.ipynb
 auto_gptq-0.4.0+cu118-cp310-cp310-linux_x86_64.whl
'BDS Presentation.gslides'
'Copy of finance_chatbot.ipynb'
 db/
 final_chatbot.ipynb
'Final Report'/
'Final Submission'/
 finance_chatbot.ipynb
 Glassdoor/
 glassdoor_text/
'Midterm Report.gdoc'
'Presentation - May 6th'/
'Project Diary - Every Effort Made Matters.gdoc'
 sec_scraping.ipynb
 test_pdfs/
 test_txts/
 tickers.csv


In [ ]:
!pip install sec-api

In [ ]:
from sec_api import QueryApi
from sec_api import RenderApi

import json
import pandas as pd
import os
import multiprocessing

API_KEY = "  "
queryApi = QueryApi(API_KEY)
renderApi = RenderApi(API_KEY)

In [ ]:
# LOAD JSON FILE OF COMPANIES IN THE S&P500 MADE BY CHRIS
data = pd.read_json('data/glassdoor/top500.json')

# EXTRACT TICKER COLUMN ONLY AND MAKE INTO LIST
ticker_list = data[['ticker']].ticker.to_list()
print(ticker_list)
print(len(ticker_list))

['WMT', 'AMZN', 'XOM', 'AAPL', 'UNH', 'CVS', 'BRKA', 'GOOGL', 'MCK', 'CVX', 'ABC', 'COST', 'MSFT', 'CAH', 'CI', 'MPC', 'PSX', 'VLO', 'F', 'HD', 'GM', 'ELV', 'JPM', 'KR', 'CNC', 'VZ', 'WBA', 'FNMA', 'CMCSA', 'T', 'META', 'BAC', 'TGT', 'DELL', 'ADM', 'C', 'UPS', 'PFE', 'LOW', 'JNJ', 'FDX', 'HUM', 'ET', None, 'FMCC', 'PEP', 'WFC', 'DIS', 'COP', 'TSLA', 'PG', 'GE', 'ACI', 'MET', 'GS', 'SYY', 'RTX', 'BA', 'SNEX', 'LMT', 'MS', 'INTC', 'HPQ', 'SNX', 'IBM', 'HCA', 'PRU', 'CAT', 'MRK', 'INT', None, 'EPD', 'ABBV', 'PAGP', 'DOW', 'AIG', 'AXP', None, 'CHTR', 'TSN', 'DE', 'CSCO', None, 'ALL', 'DAL', None, 'TJX', 'PGR', 'AAL', None, 'PFGC', 'PBF', 'NKE', 'BBY', 'BMY', 'UAL', 'TMO', 'QCOM', 'ABT', 'KO', 'ORCL', 'NUE', None, None, 'GD', 'COF', 'DINO', 'DG', 'ARW', 'OXY', None, 'TRV', 'NOC', None, 'HON', 'MMM', 'USFD', 'WBD', 'LEN', 'DHI', 'JBL', 'LNG', 'AVGO', 'KMX', 'SBUX', 'MOH', 'UBER', 'PM', 'NFLX', 'NRG', 'MDLZ', 'DHR', 'CRM', 'PARA', 'CBRE', 'MU', 'V', 'SO', 'UNFI', 'PCAR', 'DUK', 'LLY', 'HPE', 

In [ ]:
# SCRAPING THE MOST RECENT 10K REPORT FOR EACH COMPANY IN THE S&P 500
print(f"Starting download")

# WRITE URLS TO TEXT FILE
with open("data/SEC/10k_filings_url.txt", "w") as log_file:
    for tick in ticker_list:
        # QUERYING BY EACH COMPANY'S TICKER
        base_query = {
          "query": {
              "query_string": {
                  "query": f"formType:\"10-K\" AND ticker:{tick}",
                  "time_zone": "America/New_York"
              }
          },
          "from": "0",
          "size": "1", # GRABBING ONLY THE MOST RECENT ONE
          "sort": [{"filedAt": {"order": "desc"}}]  # SORTING BY "filedAt" IN DESCENDING ORDER
        }

        # USING QUERY API TO GET RESPONSE
        response = queryApi.get_filings(base_query)

        # CHECK FOR FILINGS AND PROCESS
        if len(response["filings"]) > 0:
            # SINCE 10K IS FILED ANNUALLY, SHOULD BE THE MOST RECENT ONE
            urls_list = list(map(lambda x: x["linkToFilingDetails"], response["filings"]))

            # TRANSFORM LIST OF URLS AND WRITE TO TEXT FILE
            urls_string = "\n".join(urls_list) + "\n"
            log_file.write(urls_string)

print("Finished")

In [ ]:
# SCRAPING THE FIVE MOST RECENT 8K REPORTS FOR EACH COMPANY IN THE S&P 500
print(f"Starting download")

# WRITE URLS TO TEXT FILE
with open("data/SEC/8k_filings_url.txt", "a") as log_file:
    for tick in ticker_list:
        # QUERYING BY EACH COMPANY'S TICKER
        base_query = {
          "query": {
              "query_string": {
                  "query": f"formType:\"8-K\" AND ticker:{tick}",
                  "time_zone": "America/New_York"
              }
          },
          "from": "0",
          "size": "5", # GRABBING ONLY THE FIVE MOST RECENT ONES
          "sort": [{"filedAt": {"order": "desc"}}]  # SORTING BY "filedAt" IN DESCENDING ORDER
        }

        response = queryApi.get_filings(base_query)

        # CHECK FOR FILINGS AND PROCESS
        if len(response["filings"]) > 0:
            # GRABBING ONLY MOST RECENT 8K REPORTS
            urls_list = list(map(lambda x: x["linkToFilingDetails"], response["filings"]))

            # TRANSFORM LIST OF URLS AND WRITE TO TEXT FILE
            urls_string = "\n".join(urls_list) + "\n"
            log_file.write(urls_string)

print("Finished")

In [ ]:
# download filing and save to "filings" folder
def download_filing(url):
    try:
        filing = renderApi.get_filing(url)
        file_name = url.split("/")[-2] + "-" + url.split("/")[-1]
        download_to = "./data/SEC/10k_filings/" + file_name
        with open(download_to, "w") as f:
            f.write(filing)
    except Exception as e:
        print("Problem with {url}".format(url=url))
        print(e)

# load URLs from log file
def load_urls():
    log_file = open("data/SEC/10k_filings_url.txt", "r")
    urls = log_file.read().split("\n") # convert long string of URLs into a list
    log_file.close()
    return urls

def download_all_filings():
    print("Start downloading all filings")

    download_folder = "./data/SEC/10k_filings"
    if not os.path.isdir(download_folder):
        os.makedirs(download_folder)

    urls = load_urls()
    print(f"{len(urls)} filing URLs loaded")

    number_of_processes = 1

    with multiprocessing.Pool(number_of_processes) as pool:
        pool.map(download_filing, urls)

    print("All filings downloaded")

download_all_filings()

In [ ]:
# download filing and save to "filings" folder
def download_filing(url):
    try:
        filing = renderApi.get_filing(url)
        file_name = url.split("/")[-2] + "-" + url.split("/")[-1]
        download_to = "./data/SEC/8k_filings/" + file_name
        with open(download_to, "w") as f:
            f.write(filing)
    except Exception as e:
        print("Problem with {url}".format(url=url))
        print(e)

# load URLs from log file
def load_urls():
    log_file = open("data/SEC/8k_filings_url.txt", "r")
    urls = log_file.read().split("\n") # convert long string of URLs into a list
    log_file.close()
    return urls

def download_all_filings():
    print("Start downloading all filings")

    download_folder = "./data/SEC/8k_filings"
    if not os.path.isdir(download_folder):
        os.makedirs(download_folder)

    urls = load_urls()
    print(f"{len(urls)} filing URLs loaded")

    number_of_processes = 1

    with multiprocessing.Pool(number_of_processes) as pool:
        pool.map(download_filing, urls)

    print("All filings downloaded")

In [ ]:
download_all_filings()

# EXPERIMENTING WITH OTHER EXTRACTION METHODS - DON'T RUN

In [ ]:
from sec_api import XbrlApi
xbrlApi = XbrlApi(API_KEY)

In [ ]:
# helper function to pretty print long, single-line text to multi-line text
def pprint(text, line_length=100):
    words = text.split(' ')
    lines = []
    current_line = ''
    for word in words:
      if len(current_line + ' ' + word) <= line_length:
        current_line += ' ' + word
      else:
        lines.append(current_line.strip())
        current_line = word
    if current_line:
      lines.append(current_line.strip())
    print('\n'.join(lines))

In [ ]:
with open(file_id, 'r') as file:
    for i, url in enumerate(file, start=1):
        url = url.strip()
        # Assume extractorApi is a defined API for extracting sections
        texts = [extractorApi.get_section(url, str(item), 'text') for item in range(1, 16)]  # Simplified extraction loop

        combined_text = "\n".join([text for text in texts if text])

        # Define the file name dynamically
        file_name = f"output_{i}.txt"
        full_path = os.path.join(folder_path, file_name)

        # Write the combined text to a file in the specified Google Drive folder
        with open(full_path, 'w') as output_file:
            output_file.write(combined_text)
        print(f"File {full_path} written successfully.")

In [ ]:
url_10k_2022 = 'https://www.sec.gov/Archives/edgar/data/1652044/000165204423000016/goog-20221231.htm'

xbrl_json_2022 = xbrlApi.xbrl_to_json(htm_url=url_10k_2022)

In [ ]:
xbrl_json_2022

{'CoverPage': {'DocumentType': '10-K',
  'DocumentAnnualReport': 'true',
  'DocumentPeriodEndDate': '2022-12-31',
  'CurrentFiscalYearEndDate': '--12-31',
  'DocumentTransitionReport': 'false',
  'EntityFileNumber': '001-37580',
  'EntityRegistrantName': 'Alphabet Inc.',
  'EntityIncorporationStateCountryCode': 'DE',
  'EntityTaxIdentificationNumber': '61-1767919',
  'EntityAddressAddressLine1': '1600 Amphitheatre Parkway',
  'EntityAddressCityOrTown': 'Mountain View',
  'EntityAddressStateOrProvince': 'CA',
  'EntityAddressPostalZipCode': '94043',
  'CityAreaCode': '650',
  'LocalPhoneNumber': '253-0000',
  'Security12bTitle': [{'period': {'startDate': '2022-01-01',
     'endDate': '2022-12-31'},
    'segment': {'dimension': 'us-gaap:StatementClassOfStockAxis',
     'value': 'us-gaap:CommonClassAMember'},
    'value': 'Class\xa0A Common Stock, $0.001 par value'},
   {'period': {'startDate': '2022-01-01', 'endDate': '2022-12-31'},
    'segment': {'dimension': 'us-gaap:StatementClassOfS

In [ ]:
# OLD VERSION
API_KEY = 'a900a8c75689f47512888b3cfc088c1ccae24d0b55f05b61b1af4964848ed5a6'
from sec_api import ExtractorApi

extractorApi = ExtractorApi(API_KEY)
file_id = 'filings_url.txt'
# URL of Tesla's 10-K filing
#filing_10_k_url = 'https://www.sec.gov/Archives/edgar/data/1318605/000162828024002390/tsla-20231231.htm'

import os

# Path to the Google Drive folder
folder_path = '10k_filings_txt'

# Ensure the folder exists
os.makedirs(folder_path, exist_ok=True)

all_text = ""
with open(file_id, 'r') as file:
    for i, url in enumerate(file, start=1): #Newly added
    #for url in file:
      url = url.strip()
      # extract text sections
      item_1_text    = extractorApi.get_section(url, '1', 'text')
      item_1_a_text  = extractorApi.get_section(url, '1A', 'text')
      item_1_b_text  = extractorApi.get_section(url, '1B', 'text')
      item_2_text    = extractorApi.get_section(url, '2', 'text')
      item_3_text    = extractorApi.get_section(url, '3', 'text')
      item_4_text    = extractorApi.get_section(url, '4', 'text')
      item_5_text    = extractorApi.get_section(url, '5', 'text')
      item_6_text    = extractorApi.get_section(url, '6', 'text')
      item_7_text    = extractorApi.get_section(url, '7', 'text')
      item_7_a_text  = extractorApi.get_section(url, '7A', 'text')
      item_8_text    = extractorApi.get_section(url, '8', 'text')
      item_9_text    = extractorApi.get_section(url, '9', 'text')
      item_9_a_text  = extractorApi.get_section(url, '9A', 'text')
      item_9_b_text  = extractorApi.get_section(url, '9B', 'text')
      item_10_text   = extractorApi.get_section(url, '10', 'text')
      item_11_text   = extractorApi.get_section(url, '11', 'text')
      item_12_text   = extractorApi.get_section(url, '12', 'text')
      item_13_text   = extractorApi.get_section(url, '13', 'text')
      item_14_text   = extractorApi.get_section(url, '14', 'text')
      item_15_text   = extractorApi.get_section(url, '15', 'text')

      # Combine the texts
      texts = [item_1_text, item_1_a_text, item_1_b_text, item_2_text, item_3_text,
                 item_4_text, item_5_text, item_6_text, item_7_text, item_7_a_text,
                 item_8_text, item_9_text, item_9_a_text, item_9_b_text, item_10_text,
                 item_11_text, item_12_text, item_13_text, item_14_text, item_15_text]

      combined_text = "\n".join([text for text in texts if text])

      # Define the file name dynamically
      file_name = f"output_{i}.txt"
      full_path = os.path.join(folder_path, file_name)

      # Write the combined text to a file
      with open(file_name, 'w') as output_file:
        output_file.write(combined_text)
      print(f"File {file_name} written successfully.")

# https://sec-api.io/

# GET TABLE INFO ALSO AND SAVE WITH TEXT
# SAVE INTO TEXT FILE, SAVE ALL TEXT FILES INTO FOLDER